# Full Conversational Chatbot Integration
### Combining LCEL, LangServe, LangSmith, FastAPI, and `InMemoryChatMessageHistory`

This notebook demonstrates how to build a state-of-the-art conversational chatbot combining:
- **LangSmith**: Real-time LLM tracing and observability.
- **LCEL (LangChain Expression Language)**: Declarative chain composition using pipe operator (`|`).
- **Message Trimming (`trim_messages`)**: Managing conversation memory window to prevent token overflow.
- **`InMemoryChatMessageHistory`**: Managing stateful multi-session conversation history.
- **FastAPI & LangServe**: Exposing the chatbot via production REST endpoints and interactive playground (`/chatbot/playground`).

## 1. Environment & LangSmith Setup

In [8]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Configure LangSmith Tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
if os.getenv("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
if os.getenv("LANGCHAIN_PROJECT"):
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
else:
    os.environ["LANGCHAIN_PROJECT"] = "Full-Conversational-Chatbot"

print("✅ Environment & LangSmith tracing configured.")
print(f"Project: {os.environ.get('LANGCHAIN_PROJECT')}")

✅ Environment & LangSmith tracing configured.
Project: GenAIAPPWithOPENAI


## 2. Initialize LLM Model (Groq)

In [9]:
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
model = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=groq_api_key)
print("✅ LLM Model Initialized:", model.model_name)

✅ LLM Model Initialized: openai/gpt-oss-120b


## 3. LCEL Chain & Message Trimming (`trim_messages`)
We use `trim_messages` to ensure our conversation history fits within context boundaries while retaining key system messages.

In [10]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough

# Context message trimmer
trimmer = trim_messages(
    max_tokens=300,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

# Prompt template with dynamic language support
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful, intelligent, and friendly AI assistant. "
        "Answer all questions accurately and concisely in {language}."
    ),
    MessagesPlaceholder(variable_name="messages"),
])

parser = StrOutputParser()

# Base LCEL Chain
base_chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
    | parser
)
print("✅ LCEL Chain built successfully!")

✅ LCEL Chain built successfully!


## 4. In-Memory Session Storage (`InMemoryChatMessageHistory`)
We use `InMemoryChatMessageHistory` to keep track of chat history per session ID.

In [11]:
from typing import Dict
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage

# Store dictionary holding InMemoryChatMessageHistory instances
store: Dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap base chain with RunnableWithMessageHistory
chatbot_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="messages"
)

print("✅ Stateful chatbot with InMemoryChatMessageHistory ready.")

✅ Stateful chatbot with InMemoryChatMessageHistory ready.


c:\Users\KIIT0001\langchain-graph\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### Test Chat Sessions locally in Notebook

In [12]:
# Session 1 test
config_session1 = {"configurable": {"session_id": "user_khushi"}}
res1 = chatbot_with_history.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Khushi and I am a Chief AI Engineer.")], "language": "English"},
    config=config_session1
)
print("User Khushi (Turn 1):", res1)

# Session 1 follow-up
res2 = chatbot_with_history.invoke(
    {"messages": [HumanMessage(content="What is my name and role?")], "language": "English"},
    config=config_session1
)
print("User Khushi (Turn 2):", res2)

User Khushi (Turn 1): Hello, Khushi! It’s great to meet you. How can I assist you today?
User Khushi (Turn 2): Your name is **Khushi**, and your role is **Chief AI Engineer**.


In [13]:
# Session 2 test (Separate session ID)
config_session2 = {"configurable": {"session_id": "user_john"}}
res3 = chatbot_with_history.invoke(
    {"messages": [HumanMessage(content="What is my name?")], "language": "English"},
    config=config_session2
)
print("User John (Turn 1 - Isolated context):", res3)

User John (Turn 1 - Isolated context): I’m sorry, but I don’t have any information about your name. If you’d like to share it, feel free to let me know!


## 5. Exposing Server via FastAPI & LangServe
We can host `chatbot_with_history` as a FastAPI application using `add_routes` from `langserve`.

In [14]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from langserve import add_routes
from pydantic import BaseModel, Field
from typing import Optional

app = FastAPI(
    title="LangChain Full Conversational Chatbot Server",
    version="1.0",
    description="Full Chatbot combining LCEL, InMemoryChatMessageHistory, LangSmith, and FastAPI."
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# LangServe routes
add_routes(app, chatbot_with_history, path="/chatbot")

# Custom REST endpoint
class ChatRequest(BaseModel):
    session_id: str
    message: str
    language: Optional[str] = "English"

@app.post("/api/chat")
async def chat_api(req: ChatRequest):
    cfg = {"configurable": {"session_id": req.session_id}}
    ans = chatbot_with_history.invoke(
        {"messages": [HumanMessage(content=req.message)], "language": req.language or "English"},
        config=cfg
    )
    return {"session_id": req.session_id, "response": ans}

print("✅ FastAPI & LangServe app defined. Run 'python s9_full_chatbot_server.py' to launch server on http://127.0.0.1:8000!")

✅ FastAPI & LangServe app defined. Run 'python s9_full_chatbot_server.py' to launch server on http://127.0.0.1:8000!
